# So sánh smoke M0/M1 trên cùng 20 câu
Dùng cùng tập validation, phiên bản mô hình gốc, prompt, cấu hình 4-bit, độ phân giải ảnh và cách sinh văn bản. M1 nạp adapter đã train 8 bước; notebook này chỉ suy luận, không train. Kết quả giúp kiểm tra quy trình, chưa đủ kết luận M1 tốt hơn M0. Mỗi lần chạy tạo thư mục run mới trên Drive. Output đang hiển thị thuộc lần chạy 23/09; các sửa đổi ở ô chuẩn bị sau đó chưa được chạy lại.


In [1]:
import hashlib
import json
import subprocess
import sys
import tempfile
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import torch
from google.colab import drive

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


assert torch.cuda.is_available(), "GPU runtime required"
DRIVE = Path("/content/drive/MyDrive/Vietnamese_VQA_NLE")
SOURCE = DRIVE / "source/source_20260922_161147_627608.zip"
SOURCE_HASH = "b1ca5933fb814a56ced5eb9d7c51915e46dc85fc340616146c874dd1ca2d68dd"
ARCHIVE = DRIVE / "datasets/images/coco_smoke_500.zip"
DATA_HASH = "4e0fce747dee9a1beaf36e2617deae6582115020d7c219f315525e329a4024f9"
ANNOTATIONS = DRIVE / "datasets/processed/smoke_500.jsonl"
ADAPTER = DRIVE / "runs/M1_smoke_20260922_065719/training/adapter"
for path in [
    SOURCE,
    ARCHIVE,
    ANNOTATIONS,
    ADAPTER / "adapter_config.json",
    ADAPTER / "adapter_model.safetensors",
]:
    assert path.is_file(), str(path)
assert sha256_file(SOURCE) == SOURCE_HASH
assert sha256_file(ARCHIVE) == DATA_HASH
WORK = Path(tempfile.mkdtemp(prefix="paired_", dir="/content"))


def extract_checked(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None
        assert all(
            (destination / n).resolve().is_relative_to(destination.resolve()) for n in z.namelist()
        )
        z.extractall(destination)


extract_checked(SOURCE, WORK)
REPO = WORK / "Vietnamese-VQA-Explanation"
manifest = json.loads((REPO / "SOURCE_MANIFEST.json").read_text())
for relative, digest in manifest.items():
    assert sha256_file(REPO / relative) == digest, relative
IMAGES = WORK / "coco"
extract_checked(ARCHIVE, IMAGES)
assert len(list((IMAGES / "images").glob("*.jpg"))) == 500
LOCAL_ANNOTATIONS = WORK / "smoke_500.jsonl"
LOCAL_ANNOTATIONS.write_bytes(ANNOTATIONS.read_bytes())
RUN = DRIVE / "runs" / ("paired_smoke_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f"))
RUN.mkdir(parents=True, exist_ok=False)


def write_json(name, value):
    with (RUN / name).open("x", encoding="utf-8") as f:
        json.dump(value, f, ensure_ascii=False, indent=2)


def run_cmd(label, command, *, echo=True):
    command = [str(v) for v in command]
    write_json(label + "_command.json", command)
    with (RUN / (label + ".log")).open("x", encoding="utf-8") as log:
        with subprocess.Popen(
            command,
            cwd=REPO,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        ) as process:
            for line in process.stdout:
                if echo:
                    print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
    if code:
        raise RuntimeError(f"{label} failed: exit={code}; see {RUN}")


write_json(
    "inputs.json",
    {
        "source_sha256": SOURCE_HASH,
        "images_sha256": DATA_HASH,
        "annotations_sha256": sha256_file(LOCAL_ANNOTATIONS),
        "adapter": str(ADAPTER),
        "adapter_sha256": sha256_file(ADAPTER / "adapter_model.safetensors"),
        "gpu": torch.cuda.get_device_name(0),
        "scope": "20 validation smoke only; no training",
    },
)
print("RUN:", RUN, flush=True)
run_cmd(
    "install",
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        REPO,
        "transformers==4.57.1",
        "peft==0.17.1",
        "accelerate==1.10.1",
        "bitsandbytes==0.48.1",
    ],
)
run_cmd("environment", [sys.executable, "-m", "pip", "freeze"], echo=False)
print("Environment versions saved in", RUN / "environment.log")
print("READY. No training has been run.")

Mounted at /content/drive
RUN: /content/drive/MyDrive/Vietnamese_VQA_NLE/runs/paired_smoke_20260923_003416_446950
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub

In [2]:
# Chạy tách tiến trình để giải phóng VRAM giữa M0/M1; không chạy lại trên kết quả dở dang.
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
REVISION = "66285546d2b821cf421d4f5eb2576359d3770cd3"
adapter_config = json.loads((ADAPTER / "adapter_config.json").read_text())
assert adapter_config["base_model_name_or_path"] == MODEL
assert adapter_config.get("revision") in (None, REVISION)
rows = [json.loads(s) for s in LOCAL_ANNOTATIONS.read_text().splitlines() if s.strip()]
train_images = {r["image_id"] for r in rows if r["split"] == "train"}
assert not train_images.intersection(r["image_id"] for r in rows if r["split"] == "validation")
assert len([r for r in rows if r["split"] == "validation"]) >= 20
common = [
    sys.executable,
    "-u",
    "scripts/run_m0.py",
    "--annotations",
    LOCAL_ANNOTATIONS,
    "--image-root",
    IMAGES,
    "--split",
    "validation",
    "--limit",
    "20",
    "--seed",
    "20260918",
    "--model-id",
    MODEL,
    "--revision",
    REVISION,
    "--min-pixels",
    "200704",
    "--max-pixels",
    "200704",
    "--max-new-tokens",
    "128",
    "--load-in-4bit",
]
assert not any(
    (RUN / n).exists() for n in ["m0.jsonl", "m1.jsonl", "m0_command.json", "m1_command.json"]
), "Do not overwrite or silently resume partial inference"
run_cmd("m0", common + ["--output", RUN / "m0.jsonl", "--experiment-id", "M0_MATCHED_SMOKE"])
run_cmd(
    "m1",
    common
    + ["--output", RUN / "m1.jsonl", "--experiment-id", "M1_MATCHED_SMOKE", "--adapter", ADAPTER],
)
print("Both inference processes completed:", RUN)

2026-09-23 00:35:16.647150: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!

Fetching 2 files: 100%|██████████| 2/2 [01:27<00:00, 43.70s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:26<00:00, 13.48s/it]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_V

In [3]:
left, right = [
    [json.loads(s) for s in (RUN / n).read_text().splitlines() if s.strip()]
    for n in ["m0.jsonl", "m1.jsonl"]
]
assert len(left) == len(right) == 20
a, b = [{r["question_id"]: r for r in records} for records in (left, right)]
assert len(a) == len(b) == 20 and a.keys() == b.keys()
for qid in a:
    assert a[qid]["image_sha256"] == b[qid]["image_sha256"]
    assert a[qid]["compute_dtype"] == b[qid]["compute_dtype"]
run_cmd(
    "comparison",
    [
        sys.executable,
        "scripts/compare_runs.py",
        RUN / "m0.jsonl",
        RUN / "m1.jsonl",
        "--output",
        RUN / "comparison.json",
    ],
)
report = json.loads((RUN / "comparison.json").read_text())
assert report["comparable"] and report["n"] == 20
write_json(
    "completion.json",
    {
        "status": "paired_smoke_passed",
        "n": 20,
        "m0_format_valid": sum(r["format_valid"] for r in left),
        "m1_format_valid": sum(r["format_valid"] for r in right),
        "note": (
            "Engineering smoke only. Not a final quality claim; explanations not yet evaluated."
        ),
    },
)
print(json.dumps(report, ensure_ascii=False, indent=2))
print("DONE:", RUN)

Comparable: True; issues: 0
Report: /content/drive/MyDrive/Vietnamese_VQA_NLE/runs/paired_smoke_20260923_003416_446950/comparison.json
{
  "comparable": true,
  "n": 20,
  "issues": [],
  "scores": {
    "left": {
      "exact_match": 0.35,
      "token_f1": 0.4033333333333333
    },
    "right": {
      "exact_match": 0.35,
      "token_f1": 0.4033333333333333
    }
  },
  "delta_right_minus_left": {
    "exact_match": 0.0,
    "token_f1": 0.0
  },
  "limitations": "Metadata compatibility only; not a significance test or explanation metric. Does not verify image bytes, training leakage, or truthful metadata."
}
DONE: /content/drive/MyDrive/Vietnamese_VQA_NLE/runs/paired_smoke_20260923_003416_446950
